# Lab 04.4 — Real-Time Color Detection

## Objective

Build a simple real-time detector for **red, green, blue and yellow** objects using the Logitech C270.

The processing pipeline is:

```text
camera -> BGR -> HSV -> color masks -> contours -> label
```

For reliable classroom experiments, use saturated objects such as colored paper, plastic blocks or markers.

In [ ]:
import cv2
import numpy as np
from IPython.display import display, Image, clear_output
import time

COLOR_RANGES = {
    "BLUE":   [((100, 100, 60), (130, 255, 255))],
    "GREEN":  [((35, 80, 60), (85, 255, 255))],
    "YELLOW": [((20, 100, 100), (35, 255, 255))],
    "RED":    [((0, 100, 80), (10, 255, 255)),
               ((170, 100, 80), (179, 255, 255))]
}

DRAW_COLORS = {
    "BLUE": (255, 0, 0),
    "GREEN": (0, 255, 0),
    "YELLOW": (0, 255, 255),
    "RED": (0, 0, 255)
}

In [ ]:
def make_mask(hsv, ranges):
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)

    for lower, upper in ranges:
        part = cv2.inRange(
            hsv,
            np.array(lower, dtype=np.uint8),
            np.array(upper, dtype=np.uint8)
        )
        mask = cv2.bitwise_or(mask, part)

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

## Run the detector

The detector ignores very small regions to reduce noise.

In [ ]:
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

MIN_AREA = 1200

try:
    for _ in range(150):
        ret, frame = cap.read()
        if not ret:
            print("Camera frame not available.")
            break

        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        for name, ranges in COLOR_RANGES.items():
            mask = make_mask(hsv, ranges)
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            for cnt in contours:
                area = cv2.contourArea(cnt)
                if area < MIN_AREA:
                    continue

                x, y, w, h = cv2.boundingRect(cnt)

                cv2.rectangle(frame, (x, y), (x+w, y+h), DRAW_COLORS[name], 2)
                cv2.putText(frame, name, (x, max(25, y-8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                            DRAW_COLORS[name], 2)

        ok, encoded = cv2.imencode(".jpg", frame)
        if ok:
            clear_output(wait=True)
            display(Image(data=encoded.tobytes()))

        time.sleep(0.03)

finally:
    cap.release()
    print("Camera released.")

## Investigation

Try the following:

1. show two differently colored objects simultaneously;
2. move the objects closer and farther from the camera;
3. reduce `MIN_AREA` and observe false detections;
4. change room lighting and observe the effect on the HSV thresholds.

### Question

Why is HSV segmentation usually more convenient than directly thresholding the B, G and R channels?